# Chapter 2: Handling Images with PyTorch
**Module 02: Intermediate Deep Learning with PyTorch**  
*Instructor: Michal Oleszak, Machine Learning Engineer*

> Complete PDF-integrated notebook for image representation, `ImageFolder`, augmentation, CNNs, training, and evaluation.


## Learning Objectives
- Explain grayscale and RGB image representations.
- Load class-folder image datasets with `ImageFolder`.
- Display PyTorch image tensors correctly.
- Choose label-preserving augmentations.
- Build and size a convolutional neural network.
- Train a multi-class image classifier.
- Evaluate with precision, recall, averaging strategies, and per-class analysis.


## 1. What Is an Image?
An image is a grid of pixels.

| Image Type | Pixel Value | Tensor Channels |
|---|---|---|
| Grayscale | One integer from 0 to 255 | `1` |
| RGB color | Three integers: red, green, blue, e.g. `(52, 171, 235)` | `3` |


## 2. Loading Images with `ImageFolder`
`ImageFolder` expects one folder per class.

```text
clouds_train/
  cumulus clouds/
    75cbf18.jpg
  cumulonimbus clouds/
clouds_test/
  cumulus clouds/
  cumulonimbus clouds/
```

Folder names become class names.


In [ ]:
from pathlib import Path
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

DATA_DIR = Path("datasets")
CLOUDS_ZIP = DATA_DIR / "clouds.zip"
CLOUDS_DIR = DATA_DIR / "clouds"
if CLOUDS_ZIP.exists() and not CLOUDS_DIR.exists():
    with zipfile.ZipFile(CLOUDS_ZIP) as zf:
        zf.extractall(DATA_DIR)

train_root = CLOUDS_DIR / "clouds_train"
test_root = CLOUDS_DIR / "clouds_test"
print("Cloud data available:", CLOUDS_DIR.exists())


In [ ]:
# PDF loading pipeline: parse to tensor and resize.
train_transforms_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((128, 128)),
])

if train_root.exists():
    dataset_train_basic = ImageFolder(train_root, transform=train_transforms_basic)
    dataloader_train_basic = DataLoader(dataset_train_basic, shuffle=True, batch_size=1)
    image, label = next(iter(dataloader_train_basic))
    print("Shape:", image.shape)
    print("Classes:", dataset_train_basic.class_to_idx)
else:
    print("ImageFolder pipeline defined. Extract clouds.zip or provide clouds_train to run it.")


## 3. Displaying Images
PyTorch uses channel-first tensors: `(batch, channels, height, width)`. Matplotlib expects channel-last image arrays: `(height, width, channels)`.

| Operation | Result |
|---|---|
| Batch from loader | `[1, 3, 128, 128]` |
| `squeeze()` | `[3, 128, 128]` |
| `permute(1, 2, 0)` | `[128, 128, 3]` |


In [ ]:
import matplotlib.pyplot as plt

if 'dataloader_train_basic' in globals():
    image, label = next(iter(dataloader_train_basic))
else:
    image, label = torch.rand(1, 3, 128, 128), torch.tensor([0])

print(image.shape)
image_to_show = image.squeeze().permute(1, 2, 0)
print(image_to_show.shape)
plt.imshow(image_to_show)
plt.axis("off")
plt.show()


## 4. Data Augmentation
Data augmentation generates more training variety from original images.

Benefits:
- Increases dataset diversity.
- Improves model robustness.
- Reduces overfitting.

> Choose augmentations with the data and task in mind; an augmentation that changes the label is harmful.


In [ ]:
# PDF augmentation example for cloud classification.
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(45),
    transforms.RandomAutocontrast(),
    transforms.ToTensor(),
    transforms.Resize((128, 128)),
])

test_transforms = transforms.Compose([
    # No data augmentation at test time.
    transforms.ToTensor(),
    transforms.Resize((128, 128)),
])

if train_root.exists():
    dataset_train = ImageFolder(train_root, transform=train_transforms)
    dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)
    print(dataset_train.classes)
else:
    print("Augmentation and deterministic test transforms defined.")


## 5. Why Convolutional Layers?
Linear layers on raw images are often slow, overfit easily, and do not recognize spatial patterns. Convolutional layers slide filters across the image and preserve spatial structure with fewer parameters.


## 6. CNN Building Blocks
| Layer | Purpose | PDF Example |
|---|---|---|
| `nn.Conv2d` | Learn spatial filters and feature maps | `nn.Conv2d(3, 32, kernel_size=3)` |
| Zero-padding | Preserve border information and output size | `padding=1` |
| Activation | Add non-linearity | `nn.ELU()` |
| Max pooling | Reduce spatial size by retaining maxima | `nn.MaxPool2d(kernel_size=2)` |
| Flatten | Convert feature maps to a vector | `nn.Flatten()` |


In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(),
        )
        self.classifier = nn.Linear(64 * 32 * 32, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

num_classes = len(dataset_train.classes) if 'dataset_train' in globals() else 7
net = Net(num_classes=num_classes)
print(net)
print("Output shape:", net(torch.randn(4, 3, 128, 128)).shape)


### Feature Extractor Output Size
For `128 x 128` inputs:

| Stage | Spatial Size | Channels |
|---|---:|---:|
| Input | `128 x 128` | 3 |
| Conv + ELU + Pool | `64 x 64` | 32 |
| Conv + ELU + Pool | `32 x 32` | 64 |
| Flatten | `64 * 32 * 32` | vector |

The PDF also shows `64 * 16 * 16` when the input resolution is `64 x 64`.


## 7. Training Image Classifiers
| Task Type | Loss |
|---|---|
| Binary classification | Binary cross-entropy |
| Multi-class classification | Cross-entropy |

The PDF training loop uses `nn.CrossEntropyLoss()` and `optim.Adam`.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

if 'dataloader_train' in globals():
    net.train()
    for epoch in range(1):
        total_loss = 0.0
        for images, labels in dataloader_train:
            optimizer.zero_grad()
            outputs = net(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} | loss={total_loss/len(dataloader_train):.4f}")
else:
    print("Create dataloader_train to run training.")


## 8. Evaluating Image Classifiers
Do not augment test images. Use deterministic transforms.

| Metric | Meaning |
|---|---|
| Precision | Fraction of predictions for a class that were correct |
| Recall | Fraction of true class examples recovered |

### Averaging Multi-Class Metrics
| Average | Interpretation | When Useful |
|---|---|---|
| `micro` | Global calculation | Imbalanced datasets when global performance matters |
| `macro` | Mean of per-class metrics | Care about small classes equally |
| `weighted` | Per-class mean weighted by support | Larger classes should matter more |
| `None` | One score per class | Error analysis |


In [ ]:
try:
    from torchmetrics import Precision, Recall
except ImportError:
    Precision = Recall = None

if test_root.exists():
    dataset_test = ImageFolder(test_root, transform=test_transforms)
    dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=False)
else:
    dataset_test = None

if dataset_test is not None and Precision is not None:
    metric_precision = Precision(task="multiclass", num_classes=num_classes, average="macro")
    metric_recall = Recall(task="multiclass", num_classes=num_classes, average="macro")
    net.eval()
    with torch.no_grad():
        for images, labels in dataloader_test:
            outputs = net(images)
            _, preds = torch.max(outputs, 1)
            metric_precision(preds, labels)
            metric_recall(preds, labels)
    print(f"Precision: {metric_precision.compute()}")
    print(f"Recall: {metric_recall.compute()}")
else:
    print("Install torchmetrics and provide clouds_test to run the evaluation loop.")


In [ ]:
if dataset_test is not None and Recall is not None:
    recall_metric = Recall(task="multiclass", num_classes=num_classes, average=None)
    net.eval()
    with torch.no_grad():
        for images, labels in dataloader_test:
            outputs = net(images)
            _, preds = torch.max(outputs, 1)
            recall_metric(preds, labels)
    recall = recall_metric.compute()
    per_class_recall = {k: recall[v].item() for k, v in dataset_test.class_to_idx.items()}
    print(per_class_recall)
else:
    print("Per-class analysis is ready once dataset_test and torchmetrics are available.")


## Chapter Summary
- Images are channel-based tensors; RGB images have three channels.
- `ImageFolder` loads folder-structured datasets and maps classes automatically.
- Augment only training data, and only with label-preserving transforms.
- CNNs use convolution, activation, pooling, flattening, and classifier layers.
- Precision, recall, averaging strategies, and per-class scores reveal different performance stories.
